# Homework Task 1: PySpark DataFrame + Spark SQL
This notebook solves all requested tasks in one place.
For each SELECT query, two solutions are provided: DataFrame API and Spark SQL.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Assumption
`products_df`, `sales_df`, and `sellers_df` are already available from the previous lesson.
If needed, add your data-loading cell before running the rest of this notebook.

## Task 1.1: Create TempViews

In [ ]:
products_df.createOrReplaceTempView("products")
sales_df.createOrReplaceTempView("sales")
sellers_df.createOrReplaceTempView("sellers")

print("TempViews created: products, sales, sellers")

## Task 1.2: Number of products sold at least once

In [ ]:
# DataFrame solution
products_sold_df = sales_df.select("product_id").distinct()
products_sold_count_df = products_sold_df.count()
print(f"Products sold at least once (DataFrame): {products_sold_count_df}")

In [ ]:
# Spark SQL solution
spark.sql("""
SELECT COUNT(DISTINCT product_id) AS products_sold_at_least_once
FROM sales
""").show()

## Task 1.3: Number of distinct sellers who sold any products

In [ ]:
# DataFrame solution
distinct_sellers_count_df = sales_df.select("seller_id").distinct().count()
print(f"Distinct sellers with sales (DataFrame): {distinct_sellers_count_df}")

In [ ]:
# Spark SQL solution
spark.sql("""
SELECT COUNT(DISTINCT seller_id) AS distinct_sellers
FROM sales
""").show()

## Task 1.4: Most popular product by number of orders

In [ ]:
# DataFrame solution
most_popular_product_df = (
    sales_df.groupBy("product_id")
    .agg(F.count("order_id").alias("orders_count"))
    .orderBy(F.desc("orders_count"), F.asc("product_id"))
    .limit(1)
)
most_popular_product_df.show()

In [ ]:
# Spark SQL solution
spark.sql("""
SELECT product_id, COUNT(order_id) AS orders_count
FROM sales
GROUP BY product_id
ORDER BY orders_count DESC, product_id ASC
LIMIT 1
""").show()

## Task 1.5: Number of distinct products sold per date

In [ ]:
# DataFrame solution
products_per_date_df = (
    sales_df.groupBy("date")
    .agg(F.countDistinct("product_id").alias("distinct_products_sold"))
    .orderBy("date")
)
products_per_date_df.show()

In [ ]:
# Spark SQL solution
spark.sql("""
SELECT date, COUNT(DISTINCT product_id) AS distinct_products_sold
FROM sales
GROUP BY date
ORDER BY date
""").show()

## Task 1.6: All sales made by seller_id = 7

In [ ]:
# DataFrame solution
sales_seller_7_df = sales_df.filter(F.col("seller_id") == 7)
sales_seller_7_df.show()

In [ ]:
# Spark SQL solution
spark.sql("""
SELECT *
FROM sales
WHERE seller_id = 7
""").show()

## Task 1.7: Remove `bill_raw_text` column from sales dataframe

In [ ]:
# DataFrame solution
sales_without_bill_df = sales_df.drop("bill_raw_text")
sales_without_bill_df.printSchema()

In [ ]:
# Spark SQL-style equivalent (explicit SELECT excluding column)
# Adjust this list if your schema contains additional columns.
spark.sql("""
SELECT order_id, date, seller_id, product_id, num_pieces_sold
FROM sales
""").show()

## Task 1.8: Top 10 biggest orders by `num_pieces_sold`

In [ ]:
# DataFrame solution
top_10_orders_df = sales_df.orderBy(F.desc("num_pieces_sold"), F.asc("order_id")).limit(10)
top_10_orders_df.show()

In [ ]:
# Spark SQL solution
spark.sql("""
SELECT *
FROM sales
ORDER BY num_pieces_sold DESC, order_id ASC
LIMIT 10
""").show()

## Task 1.9: Export notebook
- Jupyter: `File -> Save and Export Notebook As...` (or `Download as .ipynb`).
- Databricks: `File -> Export -> DBC Archive` or `Source File`.